In [ ]:
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    os.environ['PYTHONHASHSEED'] = str(seed)
    print(f'Seed set: {seed}')
set_seed()
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
    print('VRAM    :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
class Config:
    # ── Paths ─────────────────────────────────────────────────────────
    data_path    = '/home/user/Downloads/PEMS08.npz'
    adj_csv_path = '/home/user/Downloads/PEMS08.csv'
    num_nodes    = 170
    in_features  = 3
    seq_len      = 24    # ★ v19: was 12 — with seq=12, 6/12 blocks were pure padding!
    pred_len     = 12
    feature_idx  = 0
    train_ratio  = 0.7
    val_ratio    = 0.1

    # ── Model ─────────────────────────────────────────────────────────
    d_model      = 128   # ★ v19: was 96
    d_skip       = 256   # ★ v19: was 384 — leaner, less overfitting
    d_end        = 256
    d_time       = 64
    n_layers     = 8     # ★ v19: was 12; 8 correct blocks >> 12 half-padding blocks
    kernel_size  = 4
    adp_emb      = 20
    gcn_order    = 2     # ★ v19: was 3 — order-2 is sufficient, saves VRAM
    n_supports   = 3
    dropout      = 0.10

    # ── Training ──────────────────────────────────────────────────────
    batch_size   = 48
    lr           = 2e-3   # ★ v19: was 1e-3 — higher peak for OneCycle
    epochs       = 200
    patience     = 50
    weight_decay = 1e-4
    best_path    = 'mdgwnet_best.pt'

cfg    = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'MPGWNet v19 | d={cfg.d_model} | layers={cfg.n_layers} | seq={cfg.seq_len} | {device}')
print(f'Key fixes: seq_len 12→24, MAE loss, gated fusion, last-step skip, per-block time emb')


In [ ]:
HOUR_OFFSET = 12
DAY_OFFSET  = 288

def load_pems08(cfg):
    import pandas as pd, os
    raw  = np.load(cfg.data_path)
    data = raw['data'].astype(np.float32)
    T, N, F = data.shape
    print(f'Shape: {data.shape}')
    mean_np = data.mean(axis=0)
    std_np  = data.std(axis=0) + 1e-8
    data_norm = (data - mean_np[None]) / std_np[None]

    A_dist = None
    if os.path.exists(cfg.adj_csv_path):
        try:
            df_adj = pd.read_csv(cfg.adj_csv_path, header=None, skiprows=1)
            df_adj.columns = ['from', 'to', 'cost']
            A_raw  = np.zeros((N, N), dtype=np.float32)
            for _, row in df_adj.iterrows():
                i, j, c = int(row['from']), int(row['to']), float(row['cost'])
                if i < N and j < N:
                    A_raw[i,j] = c; A_raw[j,i] = c
            nonzero = A_raw[A_raw > 0]
            sigma   = nonzero.std() if len(nonzero) > 0 else 1.0
            A_dist  = np.exp(-(A_raw**2) / (sigma**2 + 1e-8))
            np.fill_diagonal(A_dist, 0.0)
            A_dist  = A_dist / (A_dist.sum(1, keepdims=True) + 1e-8)
            print(f'Adj: {(A_dist>0).sum()} nnz, avg_deg={(A_dist>0).sum()/N:.1f}')
        except Exception as e:
            print(f'Adj CSV failed: {e}')
            A_dist = None

    if A_dist is None:
        A_dist = np.eye(N, dtype=np.float32)
        print('WARNING: using identity adjacency')
    return data_norm, mean_np, std_np, A_dist


class TrafficDataset3Stream(Dataset):
    def __init__(self, data, seq_len, pred_len, feature_idx,
                 split_start=0, split_end=None):
        self.data     = data
        self.seq_len  = seq_len
        self.pred_len = pred_len
        self.feat_idx = feature_idx
        T = len(data)
        split_end = split_end or T
        eff_start = max(split_start, DAY_OFFSET)
        last_i    = split_end - seq_len - pred_len + 1
        self.indices = list(range(eff_start, last_i))

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        S, P = self.seq_len, self.pred_len
        rec  = self.data[i              : i+S             ].copy()
        hour = self.data[i-HOUR_OFFSET  : i-HOUR_OFFSET+S ].copy()
        day  = self.data[i-DAY_OFFSET   : i-DAY_OFFSET+S  ].copy()
        y    = self.data[i+S : i+S+P,   :, self.feat_idx  ].copy()
        tod  = np.array([(i+t) % 288      for t in range(S)], dtype=np.int64)
        dow  = np.array([((i+t)//288) % 7 for t in range(S)], dtype=np.int64)
        return (torch.from_numpy(rec),  torch.from_numpy(hour),
                torch.from_numpy(day),  torch.from_numpy(y),
                torch.from_numpy(tod),  torch.from_numpy(dow))


def build_dataloaders(cfg):
    set_seed()
    data, mean_np, std_np, A_dist = load_pems08(cfg)
    T  = len(data)
    t1 = int(T * cfg.train_ratio)
    t2 = int(T * (cfg.train_ratio + cfg.val_ratio))
    kw    = dict(batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
    ds_kw = dict(data=data, seq_len=cfg.seq_len, pred_len=cfg.pred_len,
                 feature_idx=cfg.feature_idx)
    dl_tr = DataLoader(TrafficDataset3Stream(**ds_kw, split_start=0,  split_end=t1), shuffle=True,  **kw)
    dl_va = DataLoader(TrafficDataset3Stream(**ds_kw, split_start=t1, split_end=t2), shuffle=False, **kw)
    dl_te = DataLoader(TrafficDataset3Stream(**ds_kw, split_start=t2, split_end=T),  shuffle=False, **kw)
    print(f'Train={len(dl_tr.dataset)} | Val={len(dl_va.dataset)} | Test={len(dl_te.dataset)}')
    return dl_tr, dl_va, dl_te, mean_np, std_np, A_dist

print('3-stream dataset ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# DiffusionGCN — unchanged from v18
# ══════════════════════════════════════════════════════════════════════
class DiffusionGCN(nn.Module):
    def __init__(self, d_in, d_out, n_supports=3, order=2, dropout=0.1):
        super().__init__()
        self.mlp   = nn.Linear(d_in * (1 + n_supports * order), d_out)
        self.drop  = nn.Dropout(dropout)
        self.order = order
    def forward(self, x, supports):  # x: (B*S, N, d)
        hs = [x]
        for A in supports:
            h = x
            for _ in range(self.order):
                h = torch.einsum('nm,bmd->bnd', A, h)
                hs.append(h)
        return self.drop(self.mlp(torch.cat(hs, dim=-1)))


# ══════════════════════════════════════════════════════════════════════
# WaveBlock — ★ v19: accepts te (temporal embedding) per-block
# ══════════════════════════════════════════════════════════════════════
class WaveBlock(nn.Module):
    def __init__(self, d_model, d_skip, kernel_size, dilation,
                 n_supports, gcn_order, dropout):
        super().__init__()
        self.dilation    = dilation
        self.kernel_size = kernel_size
        conv_kw = dict(kernel_size=(1, kernel_size), dilation=(1, dilation))
        self.filter_conv = nn.Conv2d(d_model, d_model, **conv_kw)
        self.gate_conv   = nn.Conv2d(d_model, d_model, **conv_kw)
        self.gcn         = DiffusionGCN(d_model, d_model, n_supports, gcn_order, dropout)
        self.bn          = nn.BatchNorm2d(d_model)
        self.drop        = nn.Dropout(dropout)
        self.skip_conv   = nn.Conv2d(d_model, d_skip,  (1,1))
        self.res_conv    = nn.Conv2d(d_model, d_model, (1,1))

    def forward(self, x, supports, te=None):  # x: (B, d, N, S)
        residual = x
        # ★ v19: inject temporal embedding at every block
        if te is not None:
            x = x + te          # te: (B, d, 1, S) broadcasts over N
        pad = (self.kernel_size - 1) * self.dilation
        x_pad = F.pad(x, [pad, 0])
        f = torch.tanh   (self.filter_conv(x_pad))
        g = torch.sigmoid(self.gate_conv  (x_pad))
        x = self.drop(f * g)
        B, dm, N, S = x.shape
        xg = x.permute(0,3,2,1).reshape(B*S, N, dm)
        xg = self.gcn(xg, supports)
        x  = xg.reshape(B,S,N,dm).permute(0,3,2,1)
        x  = self.bn(x)
        skip = self.skip_conv(x)
        x    = self.res_conv(x) + residual
        return x, skip


# ══════════════════════════════════════════════════════════════════════
# GatedPeriodFusion — ★ v19 REPLACES MultiPeriodFusion cross-attention
#
# Why: cross-attention at (B*N, S, d) is 4.26B ops per forward pass.
# More critically, it treats hour/day as generic KV — but they ARE aligned
# temporally with rec (same timestep offsets), so a learned gate is all
# that's needed to decide how much periodic context to absorb.
#
# Gate: conditioned on the FUSED feature (rec + hour + day) so each stream
# can modulate itself based on the overall context.
# ══════════════════════════════════════════════════════════════════════
class GatedPeriodFusion(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        # A lightweight MLP gate: how much of hour/day to admit
        self.gate = nn.Sequential(
            nn.Linear(d_model * 3, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model * 2),
            nn.Sigmoid()
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, h_rec, h_hour, h_day):  # all: (B, d, N, S)
        B, dm, N, S = h_rec.shape
        # (B, d, N, S) → (B*N*S, d)
        def fl(t): return t.permute(0,2,3,1).reshape(B*N*S, dm)
        r, h, d_ = fl(h_rec), fl(h_hour), fl(h_day)
        # Concat all three to compute gates
        gates = self.gate(torch.cat([r, h, d_], dim=-1))  # (B*N*S, 2*d)
        g_h, g_d = gates[:, :dm], gates[:, dm:]            # each: (B*N*S, d)
        out = self.norm(r + g_h * h + g_d * d_)            # (B*N*S, d)
        return out.reshape(B, N, S, dm).permute(0,3,1,2)   # (B, d, N, S)


print('DiffusionGCN + WaveBlock (per-block te) + GatedPeriodFusion defined.')
print('GPF replaces MultiPeriodFusion: ~20x fewer ops, no permute bugs.')


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# MPGWNet v19 — Multi-Period Graph WaveNet
#
# Architecture changes vs MD-GWNet v18:
#   1. seq_len=24 (was 12): 6/12 blocks were >50% padding-dominated
#   2. GatedPeriodFusion (was MultiPeriodFusion cross-attn): 20x cheaper
#   3. NO STFormer: 9B ops/step saved; gradient through simpler path
#   4. Per-block temporal injection: tod+dow at every WaveBlock depth
#   5. Hour/day streams get shifted temporal embeddings (correct context)
#   6. Skip aggregation: last step ONLY (was mean of last 4)
#   7. Direct MAE loss (was Huber delta=0.2 — mismatched to eval metric)
# ══════════════════════════════════════════════════════════════════════

class MPGWNet(nn.Module):
    def __init__(self, cfg, A_fwd, A_bwd):
        super().__init__()
        N, dm = cfg.num_nodes, cfg.d_model

        # ── Input projections (3 streams) ─────────────────────────────
        self.sc_rec  = nn.Conv2d(cfg.in_features, dm, (1,1))
        self.sc_hour = nn.Conv2d(cfg.in_features, dm, (1,1))
        self.sc_day  = nn.Conv2d(cfg.in_features, dm, (1,1))

        # ── Temporal embeddings ───────────────────────────────────────
        self.emb_tod   = nn.Embedding(288, cfg.d_time)
        self.emb_dow   = nn.Embedding(7,   cfg.d_time)
        self.time_proj = nn.Linear(cfg.d_time * 2, dm)
        # ★ Separate time projections for hour/day streams
        self.time_proj_h = nn.Linear(cfg.d_time * 2, dm)  # hour-shifted
        self.time_proj_d = nn.Linear(cfg.d_time * 2, dm)  # day-shifted
        self.node_emb    = nn.Parameter(torch.randn(1, dm, N, 1) * 0.01)

        # ── Adjacency ─────────────────────────────────────────────────
        self.register_buffer('A_fwd', A_fwd)
        self.register_buffer('A_bwd', A_bwd)
        self.E1 = nn.Parameter(torch.randn(N, cfg.adp_emb) * 0.01)
        self.E2 = nn.Parameter(torch.randn(cfg.adp_emb, N) * 0.01)

        # ── Gated period fusion (replaces cross-attention) ─────────────
        self.gpf = GatedPeriodFusion(dm, dropout=cfg.dropout)

        # ── ★ Correct dilation sequence: [1,2,4,8]*2 ──────────────────
        dilations = ([1, 2, 4, 8] * ((cfg.n_layers + 3) // 4))[:cfg.n_layers]
        print(f'Dilations ({len(dilations)} blocks): {dilations}')
        self.blocks = nn.ModuleList([
            WaveBlock(dm, cfg.d_skip, cfg.kernel_size, dil,
                      cfg.n_supports, cfg.gcn_order, cfg.dropout)
            for dil in dilations
        ])

        # ── Output head (no STFormer) ──────────────────────────────────
        self.out_head = nn.Sequential(
            nn.Conv2d(cfg.d_skip, cfg.d_end,    (1,1)), nn.ReLU(),
            nn.Conv2d(cfg.d_end,  cfg.pred_len, (1,1))
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)
        nn.init.normal_(self.emb_tod.weight, std=0.01)
        nn.init.normal_(self.emb_dow.weight, std=0.01)

    def _supports(self):
        A_adp = F.softmax(F.relu(self.E1 @ self.E2), dim=-1)
        return [self.A_fwd, self.A_bwd, A_adp]

    def _make_te(self, tod, dow, proj):  # → (B, d_model, 1, S)
        te = proj(torch.cat([self.emb_tod(tod),
                              self.emb_dow(dow)], dim=-1))  # (B, S, d)
        return te.permute(0,2,1).unsqueeze(2)                # (B, d, 1, S)

    def forward(self, x_rec, x_hour, x_day, tod=None, dow=None):
        # x_*: (B, S, N, F)  |  tod, dow: (B, S)
        def to4d(x): return x.permute(0,3,2,1)  # → (B, F, N, S)

        h_rec  = self.sc_rec (to4d(x_rec )) + self.node_emb
        h_hour = self.sc_hour(to4d(x_hour))
        h_day  = self.sc_day (to4d(x_day ))

        te = None
        if tod is not None and dow is not None:
            # ★ rec: use actual tod/dow
            te_rec  = self._make_te(tod, dow, self.time_proj)
            # ★ hour stream: 12 steps earlier in the day
            tod_h   = (tod - 12) % 288
            te_hour = self._make_te(tod_h, dow, self.time_proj_h)
            # ★ day stream: same time-of-day, 1 day earlier
            dow_d   = (dow - 1) % 7
            te_day  = self._make_te(tod, dow_d, self.time_proj_d)

            h_rec  = h_rec  + te_rec
            h_hour = h_hour + te_hour
            h_day  = h_day  + te_day
            te = te_rec   # used for per-block injection

        # ── Gated period fusion ────────────────────────────────────────
        x = self.gpf(h_rec, h_hour, h_day)   # (B, d, N, S)

        # ── WaveBlocks with per-block temporal embedding ──────────────
        sup = self._supports()
        skip_acc = 0
        for blk in self.blocks:
            x, skip = blk(x, sup, te=te)
            skip_acc = skip_acc + skip

        # ★ v19 FIX: last timestep only (was mean of last 4)
        out = self.out_head(F.relu(skip_acc[:,:,:,-1:]))  # (B, pred_len, N, 1)
        return out.squeeze(-1)                             # (B, pred_len, N)


def build_model(cfg, A_dist_np, device):
    set_seed()
    A_fwd = torch.from_numpy(A_dist_np  ).float().to(device)
    A_bwd = torch.from_numpy(A_dist_np.T).float().to(device)
    A_fwd = A_fwd / (A_fwd.sum(1, keepdim=True) + 1e-8)
    A_bwd = A_bwd / (A_bwd.sum(1, keepdim=True) + 1e-8)
    model = MPGWNet(cfg, A_fwd, A_bwd).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'MPGWNet v19 | {n_params:,} parameters')
    # Smoke test
    with torch.no_grad():
        B = 2; S = cfg.seq_len; N = cfg.num_nodes; F = cfg.in_features
        dummy = lambda: torch.randn(B, S, N, F).to(device)
        tod = torch.randint(0, 288, (B, S)).to(device)
        dow = torch.randint(0, 7,   (B, S)).to(device)
        out = model(dummy(), dummy(), dummy(), tod, dow)
        ok  = list(out.shape) == [B, cfg.pred_len, N]
        print(f'Smoke test: {out.shape}  {chr(10003) if ok else "FAIL"}')
    return model

print('MPGWNet v19 defined.')


In [ ]:
dl_train, dl_val, dl_test, mean_np, std_np, A_dist_np = build_dataloaders(cfg)
mean_flow = torch.from_numpy(mean_np[:, cfg.feature_idx]).to(device)
std_flow  = torch.from_numpy(std_np [:, cfg.feature_idx]).to(device)
print(f'Batches: Train={len(dl_train)} Val={len(dl_val)} Test={len(dl_test)}')

model = build_model(cfg, A_dist_np, device)

In [ ]:
# ★ v19: Use masked MAE as training loss (was Huber delta=0.2)
# Training directly on MAE = optimising exactly what we evaluate.
# Masking: exclude zero-flow sensors to avoid bias-to-zero.

def masked_mae(pred, target, null_val=0.0):
    mask = (target.abs() > null_val).float()
    return (torch.abs(pred - target) * mask).sum() / (mask.sum() + 1e-8)

def masked_rmse(pred, target, null_val=0.0):
    mask = (target.abs() > null_val).float()
    return (((pred - target)**2 * mask).sum() / (mask.sum() + 1e-8)).sqrt()

def masked_mape(pred, target, mask_val=10.0):
    # Only compute MAPE for sensors with flow > 10 (avoids div-by-zero)
    mask = (target.abs() > mask_val).float()
    if mask.sum() < 1: return torch.tensor(0.0, device=pred.device)
    return (torch.abs((pred - target) / (target.abs() + 1.0)) * mask).sum() / mask.sum() * 100

print('Metrics defined — masked MAE used as training loss.')


In [ ]:
scaler = torch.amp.GradScaler('cuda')

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total = 0.0
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device,  non_blocking=True)
        x_hour = x_hour.to(device, non_blocking=True)
        x_day  = x_day.to(device,  non_blocking=True)
        y      = y.to(device,      non_blocking=True)
        tod    = tod.to(device,    non_blocking=True)
        dow    = dow.to(device,    non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, x_hour, x_day, tod, dow)
            loss = masked_mae(pred, y)   # ★ direct MAE — train on what we evaluate
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 3.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader, device, mean_flow, std_flow):
    model.eval()
    maes, rmses, mapes = [], [], []
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device,  non_blocking=True)
        x_hour = x_hour.to(device, non_blocking=True)
        x_day  = x_day.to(device,  non_blocking=True)
        y      = y.to(device,      non_blocking=True)
        tod    = tod.to(device,    non_blocking=True)
        dow    = dow.to(device,    non_blocking=True)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, x_hour, x_day, tod, dow)
        pd_ = pred.float() * std_flow[None,None,:] + mean_flow[None,None,:]
        yd_ = y.float()    * std_flow[None,None,:] + mean_flow[None,None,:]
        maes.append (masked_mae (pd_, yd_).item())
        rmses.append(masked_rmse(pd_, yd_).item())
        mapes.append(masked_mape(pd_, yd_).item())
    return np.mean(maes), np.mean(rmses), np.mean(mapes)

print('Train/eval ready (MAE loss, AMP, per-batch OneCycleLR).')


In [ ]:
set_seed()
model = build_model(cfg, A_dist_np, device)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr           = cfg.lr,
    epochs           = cfg.epochs,
    steps_per_epoch  = len(dl_train),
    pct_start        = 0.08,
    anneal_strategy  = 'cos',
    div_factor       = 10.0,
    final_div_factor = 1000.0,
)

best_val_mae  = float('inf')
patience_cnt  = 0
history       = {'train_loss':[], 'val_mae':[], 'val_rmse':[], 'val_mape':[]}

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'MPGWNet v19 | {params:,} params')
print(f'OneCycleLR peak={cfg.lr} | epochs={cfg.epochs} | patience={cfg.patience}')
print(f'3-stream (rec+hour+day) → GatedFusion → 8 WaveBlocks → MAE head')
print(f'Baseline -> MAE=13.114 | RMSE=22.623 | MAPE=8.471%')
print('='*70)

for epoch in range(1, cfg.epochs+1):
    tr_loss = train_epoch(model, dl_train, optimizer, scheduler, device)
    val_mae, val_rmse, val_mape = eval_epoch(
        model, dl_val, device, mean_flow, std_flow)

    history['train_loss'].append(tr_loss)
    history['val_mae'].append(val_mae)
    history['val_rmse'].append(val_rmse)
    history['val_mape'].append(val_mape)

    tag = ''
    if val_mae < best_val_mae:
        best_val_mae  = val_mae
        patience_cnt  = 0
        torch.save(model.state_dict(), cfg.best_path)
        tag = '  <- best'
    else:
        patience_cnt += 1
        if patience_cnt >= cfg.patience:
            print(f'Early stop at epoch {epoch}.')
            break

    lr_now = optimizer.param_groups[0]['lr']
    beat   = ' *** BEAT BASELINE!' if val_mae < 13.114 else ''
    if epoch % 5 == 0 or tag:
        print(f'Ep {epoch:03d} | Loss={tr_loss:.4f} | '
              f'MAE={val_mae:.3f} RMSE={val_rmse:.3f} MAPE={val_mape:.2f}% '
              f'lr={lr_now:.1e}{tag}{beat}')

print(f'\nBest Val MAE: {best_val_mae:.3f}  (baseline: 13.114)')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train_loss'], color='steelblue')
axes[0].set_title('Train Loss (Huber)'); axes[0].set_xlabel('Epoch')
axes[1].plot(history['val_mae'], color='tab:orange', label='Val MAE')
axes[1].axhline(13.114, color='red', ls='--', label='Baseline 13.114')
axes[1].set_title('Validation MAE'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[2].plot(history['val_rmse'], color='tab:green', label='Val RMSE')
axes[2].axhline(22.623, color='red', ls='--', label='Baseline 22.623')
axes[2].set_title('Validation RMSE'); axes[2].set_xlabel('Epoch'); axes[2].legend()
plt.tight_layout(); plt.savefig('training_curves_v21.png', dpi=150); plt.show()

In [ ]:
model.load_state_dict(torch.load(cfg.best_path, map_location=device))

@torch.no_grad()
def paper_eval(model, loader, device, mean_flow, std_flow):
    model.eval()
    all_pred, all_true = [], []
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device);   x_hour = x_hour.to(device)
        x_day  = x_day.to(device);   y      = y.to(device)
        tod    = tod.to(device);      dow    = dow.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, x_hour, x_day, tod, dow)
        pd_ = pred.float() * std_flow[None,None,:] + mean_flow[None,None,:]
        yd_ = y.float()    * std_flow[None,None,:] + mean_flow[None,None,:]
        all_pred.append(pd_.cpu()); all_true.append(yd_.cpu())

    P = torch.cat(all_pred); Y = torch.cat(all_true)
    mae  = torch.abs(P-Y).mean().item()
    rmse = ((P-Y)**2).mean().sqrt().item()
    mask = Y.abs() > 10.0
    mape = (torch.abs((P[mask]-Y[mask])/(Y[mask].abs()+1.0))).mean().item()*100

    print('\n' + '='*60)
    print('  TEST RESULTS — MPGWNet v19 — all 12 steps')
    print('='*60)
    print(f'  MAE  : {mae:.3f}   baseline: 13.114   d={mae-13.114:+.3f}')
    print(f'  RMSE : {rmse:.3f}   baseline: 22.623   d={rmse-22.623:+.3f}')
    print(f'  MAPE : {mape:.3f}%  baseline:  8.471%  d={mape-8.471:+.3f}%')
    beaten = mae < 13.114
    print(f'\n  Baseline BEATEN: {"YES *** " if beaten else "NOT YET"}')
    print('='*60)
    return mae, rmse, mape

paper_eval(model, dl_test, device, mean_flow, std_flow)


In [ ]:
@torch.no_grad()
def horizon_eval(model, loader, device, mean_flow, std_flow):
    model.eval()
    buf = {h:{'mae':[],'rmse':[],'mape':[]} for h in [2,5,11]}
    for x_rec, x_hour, x_day, y, tod, dow in loader:
        x_rec  = x_rec.to(device);   x_hour = x_hour.to(device)
        x_day  = x_day.to(device);   y      = y.to(device)
        tod    = tod.to(device);      dow    = dow.to(device)
        with torch.amp.autocast('cuda'):
            pred = model(x_rec, x_hour, x_day, tod, dow)
        pd_ = pred.float()*std_flow[None,None,:]+mean_flow[None,None,:]
        yd_ = y.float()   *std_flow[None,None,:]+mean_flow[None,None,:]
        for h in buf:
            buf[h]['mae'].append (masked_mae (pd_[:,h,:], yd_[:,h,:]).item())
            buf[h]['rmse'].append(masked_rmse(pd_[:,h,:], yd_[:,h,:]).item())
            buf[h]['mape'].append(masked_mape(pd_[:,h,:], yd_[:,h,:]).item())
    print(f"\n{'Horizon':>14} | {'MAE':>8} | {'RMSE':>8} | {'MAPE':>9}")
    print('-'*52)
    for h, lbl in zip([2,5,11], ['3-step(15m)','6-step(30m)','12-step(60m)']):
        m = {k: np.mean(v) for k,v in buf[h].items()}
        print(f"{lbl:>14} | {m['mae']:>8.3f} | {m['rmse']:>8.3f} | {m['mape']:>8.2f}%")

horizon_eval(model, dl_test, device, mean_flow, std_flow)